# v3, v4 학습을 위한 데이터 전처리 파일
## D+F+U 3각도만 뽑아서 새 파일로 저장

In [ ]:
# 셀 1: GCS 인증 
from google.colab import auth
auth.authenticate_user()

PROJECT_ID  = 'project-id'
BUCKET_NAME = 'sign-language-data-2026'

!gcloud config set project {PROJECT_ID}
print(' GCS 인증 완료')

In [ ]:
# 셀 2: 라이브러리 및 GCS 연결 
import numpy as np
import json
import io
from google.cloud import storage
from collections import Counter

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

# label_map 로드
blob = bucket.blob('processed/label_map.json')
label_map = json.loads(blob.download_as_text())
print(f'label_map 로드 완료 {len(label_map)}개 단어')

In [ ]:
# 셀 3: 기존 X_train.npy 로드 
print('GCS에서 X_train.npy 로드 중... (잠깐 걸려요)')

def load_npy(blob_name): # 5각도 전체 데이터를 GCS에서 로드
    buf = io.BytesIO(bucket.blob(blob_name).download_as_bytes())
    return np.load(buf)

X_full = load_npy('processed/X_train.npy')  # (15000, 30, 274)
y_full = load_npy('processed/y_train.npy')  # (15000,)

print(f'X_full shape: {X_full.shape}')
print(f'y_full shape: {y_full.shape}')

GCS에서 X_train.npy 로드 중... (잠깐 걸려요)
X_full shape: (15000, 30, 274)
y_full shape: (15000,)


In [ ]:
# 셀 4: D+F+U 슬라이싱 
# 저장 순서: D(0), F(1), L(2), R(3), U(4) 

indices = []
for i in range(3000):
    base = i * 5
    indices += [base+0, base+1, base+4]  # D, F, U

X = X_full[indices]  # (9000, 30, 274)
y = y_full[indices]  # (9000,)

print(f'\n슬라이싱 완료')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'클래스 수: {len(np.unique(y))}')

# 검증: 단어당 샘플 수 확인
cnt = Counter(y)
print(f'단어당 샘플 수 (최소/최대): {min(cnt.values())} / {max(cnt.values())}')

In [ ]:
# 셀 5: GCS 업로드 
def upload_npy(arr, blob_name):
    buf = io.BytesIO()
    np.save(buf, arr)
    buf.seek(0)
    bucket.blob(blob_name).upload_from_file(buf, content_type='application/octet-stream')
    print(f'   {blob_name}')

print('GCS 업로드 중...')
upload_npy(X, 'processed/X_train_DFU.npy')
upload_npy(y, 'processed/y_train_DFU.npy')
print('\n업로드 완료!')